In [ ]:
import ast
import random
import operator
import requests
import json

import os
os.chdir('/home/maki')

In [ ]:
# ─── Брутфорс-солвер ──────────────────────────────────────────────────────────

def solve_countdown(nums, target):
    ops = [operator.add, operator.sub, operator.mul, operator.truediv]
    op_syms = {operator.add:'+', operator.sub:'-',
               operator.mul:'*', operator.truediv:'/'}

    def solve(numbers):
        if len(numbers) == 1:
            val, expr = numbers[0]
            if abs(val - target) < 1e-9 and abs(val - round(val)) < 1e-9:
                return expr
            return None
        for i in range(len(numbers)):
            for j in range(len(numbers)):
                if i == j: continue
                a_val, a_expr = numbers[i]
                b_val, b_expr = numbers[j]
                rest = [numbers[k] for k in range(len(numbers))
                        if k != i and k != j]
                for op in ops:
                    if op == operator.truediv and abs(b_val) < 1e-9: continue
                    if op == operator.truediv and abs(a_val % b_val) > 1e-9: continue
                    result = op(a_val, b_val)
                    sym = op_syms[op]
                    if op in (operator.mul, operator.truediv):
                        a_str = f"({a_expr})" if any(s in a_expr for s in ('+','-')) else a_expr
                        b_str = f"({b_expr})" if any(s in b_expr for s in ('+','-')) else b_expr
                    else:
                        a_str = a_expr
                        b_str = (f"({b_expr})" if op == operator.sub
                                 and any(s in b_expr for s in ('+','-')) else b_expr)
                    found = solve(rest + [(result, f"{a_str} {sym} {b_str}")])
                    if found is not None:
                        return found
        return None

    return solve([(float(n), str(n)) for n in nums])

# ─── Парсер выражений → шаги ──────────────────────────────────────────────────

def parse_expr_to_steps(expr_str):
    """
    Разбирает выражение на промежуточные шаги.
    Возвращает список (подвыражение, результат).
    """
    tree = ast.parse(expr_str, mode='eval')
    steps = []

    def evaluate(node):
        if isinstance(node, ast.Constant):
            return node.value, str(int(node.value))

        elif isinstance(node, ast.BinOp):
            left_val,  left_str  = evaluate(node.left)
            right_val, right_str = evaluate(node.right)

            op_map = {
                ast.Add:  ('+', lambda a,b: a+b),
                ast.Sub:  ('-', lambda a,b: a-b),
                ast.Mult: ('*', lambda a,b: a*b),
                ast.Div:  ('/', lambda a,b: a/b),
            }
            op_sym, op_fn = op_map[type(node.op)]
            result = op_fn(left_val, right_val)

            def needs_parens(child, parent_op):
                if not isinstance(child, ast.BinOp): return False
                if type(parent_op) in (ast.Mult, ast.Div):
                    if type(child.op) in (ast.Add, ast.Sub): return True
                return False

            l_str = f"({left_str})"  if needs_parens(node.left,  node.op) else left_str
            r_str = f"({right_str})" if needs_parens(node.right, node.op) else right_str
            expr_repr = f"{l_str} {op_sym} {r_str}"

            result_fmt = int(result) if result == int(result) else round(result, 4)
            steps.append((expr_repr, result_fmt))
            return result, expr_repr

        elif isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
            val, s = evaluate(node.operand)
            return -val, f"-{s}"

    evaluate(tree.body)
    return steps

# ─── Генератор CoT ────────────────────────────────────────────────────────────

def make_cot(nums, target, solution):
    """
    Строит CoT из реальных шагов вычисления выражения.
    """
    try:
        steps = parse_expr_to_steps(solution)
    except Exception:
        # Fallback если парсинг не удался
        return (
            f"<think>\n"
            f"Let me find a solution using {nums} to reach {target}.\n\n"
            f"After trying different combinations:\n"
            f"{solution} = {target} ✓\n"
            f"</think>\n"
            f"<answer>\n{solution}\n</answer>"
        )

    # Убираем финальный шаг — он будет в выводе отдельно
    intermediate = steps[:-1]

    # Вступление — несколько вариантов для разнообразия
    intros = [
        (f"I need to use the numbers {nums} to make {target}.\n\n"
         f"Since the target is {target}, let me think about which operations could work."),

        (f"Let me find a combination of {nums} that equals {target}.\n\n"
         f"I will try different arithmetic operations systematically."),

        (f"Target: {target}, available numbers: {nums}.\n\n"
         f"Let me explore combinations using +, -, *, /."),

        (f"I need to reach {target} using each of {nums} exactly once.\n\n"
         f"Let me think about possible combinations."),
    ]
    intro = random.choice(intros)

    # Промежуточные шаги
    if intermediate:
        steps_text = "\n\nLet me try this approach:\n"
        for expr, val in intermediate:
            steps_text += f"- {expr} = {val}\n"
    else:
        steps_text = "\n\nLet me try directly:\n"

    # Финал
    outro = (
        f"\nThis gives us:\n"
        f"{solution} = {target} ✓\n\n"
        f"All numbers {nums} used exactly once."
    )

    return (
        f"<think>\n{intro}{steps_text}{outro}\n</think>\n"
        f"<answer>\n{solution}\n</answer>"
    )

def make_user_prompt(nums, target):
    return (
        f"Using the numbers {nums}, create an equation that equals {target}. "
        f"You can use basic arithmetic operations (+, -, *, /) "
        f"and each number can only be used once. Show your work in "
        f"<think> </think> tags. And return the final equation and answer "
        f"in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>."
    )

In [ ]:
def query_server(nums, target):
    SYSTEM_PROMPT = "You are a helpful assistant. You first think about the reasoning process in the mind and then provide the user with the answer."
    
    nums = ex["nums"]
    target = ex["target"]

    FLOW_PROMPT = f"""
    The task was given: 'using the numbers {nums}, create an equation that equals {target}. 
    You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. 
    And return the final equation and answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 = 1 </answer>.'
    
    The assistant's response was:
    '{make_cot(nums, target, solve_countdown(nums, target))}'
    
    Explain the solution.
    """
    
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n"
        f"{FLOW_PROMPT}"
        f"<|im_end|>\n<|im_start|>assistant\n"
    )

    for attempt in range(5):
        try:
            resp = requests.post(
                "http://localhost:8080/completion",
                json={"prompt": prompt, "n_predict": 1024,
                      "temperature": 0.6, "top_k": 20, "top_p": 0.95,
                      "stop": ["<|im_end|>"]},
                timeout=500
            )
            generated = resp.json().get("content", "")
            # Собираем полный ответ ассистента
            assistant_content = generated.strip()
            return {
                "target"  : target,
                "nums"    : nums,
                "messages": [
                    {"role": "system",    "content": SYSTEM_PROMPT},
                    {"role": "user",      "content": make_user_prompt(nums, target)},
                    {"role": "assistant", "content": assistant_content},
                ]
            }
        except Exception as e:
            if attempt == CFG["n_samples"] - 1:
                pass 
    return None  # все попытки провалились

In [ ]:
def run_generation(tasks):
    output_path = Path("./data/countdown_qwen3_4b.jsonl")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    checkpoint_path = Path("./data/checkpoint.jsonl")

    # Загружаем уже сгенерированные (resume)
    done_keys = set()
    results = []
    if checkpoint_path.exists():
        with open(checkpoint_path) as f:
            for line in f:
                ex = json.loads(line)
                done_keys.add((ex["target"], tuple(sorted(ex["nums"]))))
                results.append(ex)
        print(f"\nResume: уже есть {len(results)} примеров в чекпоинте")

    # Фильтруем уже сделанные
    tasks_todo = [
        t for t in tasks
        if (t["target"], tuple(sorted(t["nums"]))) not in done_keys
    ]
    print(f"Осталось задач: {len(tasks_todo)}")

    n_correct = len(results)
    n_failed  = 0
    start     = time.time()

    with open(checkpoint_path, "a") as ckpt_f:
        with concurrent.futures.ThreadPoolExecutor(
            max_workers=8
        ) as executor:
            futures = {
                executor.submit(query_server, t["nums"], t["target"]): t
                for t in tasks_todo
            }
            for i, future in enumerate(
                concurrent.futures.as_completed(futures), 1
            ):
                result = future.result()
                if result is not None:
                    n_correct += 1
                    results.append(result)
                    ckpt_f.write(json.dumps(result, ensure_ascii=False) + "\n")
                    if n_correct % 500 == 0:
                        ckpt_f.flush()
                else:
                    n_failed += 1

                # Прогресс
                if i % 100 == 0 or i == len(futures):
                    elapsed = time.time() - start
                    speed   = i / elapsed if elapsed > 0 else 0
                    eta     = (len(futures) - i) / speed if speed > 0 else 0
                    rate    = n_correct / (n_correct + n_failed) * 100 if (n_correct + n_failed) > 0 else 0
                    print(
                        f"  [{i:>6}/{len(futures)}] "
                        f"correct={n_correct} "
                        f"failed={n_failed} "
                        f"rate={rate:.1f}% "
                        f"speed={speed:.1f}/s "
                        f"ETA={eta/60:.1f}m"
                    )

    # Финальное сохранение
    print(f"\nСохраняем {len(results)} примеров в {output_path}...")
    with open(output_path, "w") as f:
        for ex in results:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")

    # Статистика
    dist = Counter(len(ex["nums"]) for ex in results)
    print(f"ИТОГО: {len(results)} примеров")
    print(f"Распределение по длинам:")
    for k in sorted(dist):
        print(f"  {k} чисел: {dist[k]:>6} ({dist[k]/len(results)*100:.1f}%)")
    print(f"{'='*50}")

    return results

In [ ]:
tasks = []

with open('generated-data/data/synthetic_5_6.jsonl', 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            tasks.append(json.loads(line))

In [ ]:
run_generation(tasks)